# USER-bge-m3: base (3 ключевые метрики)

Оценка `deepvk/USER-bge-m3` (zero-shot). Fine-tuned версия пока не обучена — слот для неё закомментирован в конфиге MODELS. Метрики: Spearman ρ, Pearson r, MRR.

In [1]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали из подпапки
import os
from pathlib import Path
_p = Path.cwd()
while _p.name != 'thesis' and _p.parent != _p:
    _p = _p.parent
if _p.name == 'thesis':
    os.chdir(_p)
print('CWD:', Path.cwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
import warnings
warnings.filterwarnings('ignore')

import json
import html
import time
import gc
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
from scipy import stats
from IPython.display import HTML, display

# sentence-transformers 5.x ↔ transformers 4.57 совместимость
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import lancedb

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')


DEVICE: cpu


## Метрики

Оцениваем bi-encoder'ы по трём ключевым показателям на `golden_eval.parquet` (1598 пар с gold-скорами релевантности):

1. **Spearman ρ** — ранговая корреляция gold и cosine. Главная метрика: правильно ли модель упорядочивает пары по степени релевантности.
2. **Pearson r** — линейная корреляция. Показывает калибровку: ложатся ли точки `(gold, cos)` на прямую.
3. **MRR** — для каждого `desc_i` ищется свой `post_i` среди всех 1598 постов; средний обратный ранг. Прокси для retrieval-качества без отдельной разметки «правильных» постов.

Ниже — визуальная таблица top-5 на реальном индексе 50k. Это **не метрика**, а качественный контроль: на неё не опираемся при оценке качества.

In [3]:
# ==================== МОДЕЛИ ДЛЯ СРАВНЕНИЯ ====================
# ВНИМАНИЕ: fine-tuned версии USER-bge-m3 пока нет — показываем только base.
# Когда модель и таблица 'bge-m3-fine-tuned-50k' появятся, раскомментируй второй элемент.
MODELS = [
    {
        'key':          'bge_base',
        'display_name': 'USER-bge-m3 base',
        'model_path':   'deepvk/USER-bge-m3',
        'table_name':   'bge-m3-base-50k',
        'doc_prefix':   '',
        'query_prefix': '',
        'color':        '#d3f9d8',
    },
    # {
    #     'key':          'bge_ft',
    #     'display_name': 'USER-bge-m3 fine-tuned',
    #     'model_path':   'models/bi-encoder-bge-m3-finetuned',
    #     'table_name':   'bge-m3-fine-tuned-50k',
    #     'doc_prefix':   '',
    #     'query_prefix': '',
    #     'color':        '#8ce99a',
    # },
]


In [4]:
# --------- общие параметры ---------
LANCEDB_PATH    = './lancedb_store'
EVAL_PARQUET    = 'data/golden/golden_eval.parquet'
GT_POSTS_JSON   = 'ground_truth_posts.json'
GT_PAIRS_JSON   = 'ground_truth_pairs.json'
TOP_K_VISUAL    = 5     # сколько постов показать в визуальной таблице
BATCH_SIZE      = 64    # для encode на golden_eval (1598 строк)

print(f'Будет сравниваться моделей: {len(MODELS)}')
for m in MODELS:
    print(f"  - {m['display_name']:30s} ({m['table_name']})")


Будет сравниваться моделей: 1
  - USER-bge-m3 base               (bge-m3-base-50k)


In [5]:
# Проверяем, что все нужные LanceDB-таблицы существуют (нужно только для визуала top-5)
db = lancedb.connect(LANCEDB_PATH)
available = set(db.table_names())
missing = [m for m in MODELS if m['table_name'] not in available]
if missing:
    msg = '\n'.join(f"  - {m['display_name']}: нет таблицы {m['table_name']}" for m in missing)
    raise RuntimeError(
        f'В {LANCEDB_PATH} отсутствуют таблицы для следующих моделей:\n{msg}\n\n'
        f'Сначала прогони thesis/db/create-all-dbs.ipynb для нужных моделей, '
        f'либо закомментируй их в MODELS выше.'
    )
print(f'Все {len(MODELS)} таблиц на месте.')

# Заодно проверяем golden_eval.parquet и GT-файлы (GT нужны только для визуала)
for path in [EVAL_PARQUET, GT_POSTS_JSON, GT_PAIRS_JSON]:
    assert os.path.exists(path), f'Нет файла: {path}'
print('golden_eval.parquet, ground_truth_*.json — на месте.')


Все 1 таблиц на месте.
golden_eval.parquet, ground_truth_*.json — на месте.


In [6]:
# Загружаем golden_eval.parquet (для метрик) и ground_truth_pairs.json (для визуала)
from datasets import load_dataset

eval_ds = load_dataset('parquet', data_files=EVAL_PARQUET, split='train')
descriptions = list(eval_ds['product_desc'])
posts        = list(eval_ds['post_text'])
gold_scores  = np.array(eval_ds['score'], dtype=float)
print(f'golden_eval: {len(descriptions):,} пар')

with open(GT_POSTS_JSON, encoding='utf-8') as f:
    gt_posts = json.load(f)
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    gt_pairs = json.load(f)
post_text_by_num = {p['post_id']: p['text'] for p in gt_posts}
print(f'GT (для визуала): {len(gt_pairs)} пар, {len(gt_posts)} эталонных постов')


golden_eval: 1,598 пар
GT (для визуала): 19 пар, 10 эталонных постов


In [7]:
# Функция: считает три ключевые метрики bi-encoder'а на golden_eval.parquet
# 1) Spearman ρ  — ранговая корреляция gold и cosine (главная — качество ранжирования)
# 2) Pearson r   — линейная корреляция gold и cosine (калибровка)
# 3) MRR         — retrieval: для каждого desc_i ищется post_i среди всех 1598 постов
def compute_metrics(model, model_cfg, descriptions, posts, gold_scores):
    desc_in = [model_cfg['query_prefix'] + d for d in descriptions] if model_cfg['query_prefix'] else descriptions
    post_in = [model_cfg['doc_prefix']   + p for p in posts]        if model_cfg['doc_prefix']   else posts

    desc_embs = model.encode(desc_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)
    post_embs = model.encode(post_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)

    # Корреляции на парах (i, i)
    pair_scores = (desc_embs * post_embs).sum(dim=1).cpu().numpy()
    spearman_r, _ = stats.spearmanr(gold_scores, pair_scores)
    pearson_r, _  = stats.pearsonr(gold_scores, pair_scores)

    # MRR: для каждого desc_i — ранг post_i среди 1598 кандидатов (пропускаем пары с gold<=0)
    sim_matrix = cos_sim(desc_embs, post_embs).cpu().numpy()
    rrs = []
    for i in range(len(descriptions)):
        if gold_scores[i] <= 0:
            continue
        order = np.argsort(-sim_matrix[i])
        rank = int(np.where(order == i)[0][0]) + 1
        rrs.append(1.0 / rank)

    return {
        'spearman': spearman_r,
        'pearson':  pearson_r,
        'mrr':      float(np.mean(rrs)) if rrs else 0.0,
        'eval_pairs': len(rrs),
    }


In [8]:
# Главный цикл: загружаем модели, считаем метрики
results = OrderedDict()
loaded_models = OrderedDict()  # нужны ниже для визуала

for cfg in MODELS:
    print(f"\n--- {cfg['display_name']} ({cfg['model_path']}) ---")
    t0 = time.time()
    model = SentenceTransformer(cfg['model_path'], device=DEVICE)
    print(f'  загружена за {time.time()-t0:.0f}с, dim={model.get_sentence_embedding_dimension()}')

    t0 = time.time()
    m = compute_metrics(model, cfg, descriptions, posts, gold_scores)
    print(f'  метрики посчитаны за {time.time()-t0:.0f}с')
    print(f'    Spearman={m["spearman"]:.4f}  Pearson={m["pearson"]:.4f}  MRR={m["mrr"]:.4f}')

    results[cfg['key']] = {'cfg': cfg, 'metrics': m}
    loaded_models[cfg['key']] = model

print('\nГотово.')



--- USER-bge-m3 base (deepvk/USER-bge-m3) ---
  загружена за 5с, dim=1024
  метрики посчитаны за 724с
    Spearman=0.0322  Pearson=0.2456  MRR=0.1148

Готово.


## Сводная таблица метрик

In [9]:
# Сводная таблица: Spearman, Pearson, MRR
rows = []
for key, r in results.items():
    m = r['metrics']
    rows.append({
        'Модель':     r['cfg']['display_name'],
        'Spearman ρ': round(m['spearman'], 4),
        'Pearson r':  round(m['pearson'],  4),
        'MRR':        round(m['mrr'],      4),
    })
df = pd.DataFrame(rows)
eval_n = next(iter(results.values()))['metrics']['eval_pairs']
print(f'Метрики на golden_eval.parquet ({eval_n} пар с gold>0)')
display(df.style.background_gradient(cmap='YlGn', subset=['Spearman ρ', 'Pearson r', 'MRR']))


Метрики на golden_eval.parquet (1034 пар с gold>0)


,Модель,Spearman ρ,Pearson r,MRR
0,USER-bge-m3 base,0.032200,0.245600,0.114800


## Визуал: top-5 по каждой модели для каждой GT-пары (качественный контроль)

In [10]:
# Вспомогательная функция для визуала: экранирование + рендер одной пары
def esc(s):
    return html.escape(str(s)).replace('\n', '<br>')


def render_pair_topk(idx, total, pair, target_text, model_topk, top_k=5):
    target_norm = target_text.strip()

    col_headers = ''.join(
        f'<th style="padding:10px;text-align:left;background:{cfg["color"]};'
        f'border-bottom:2px solid #495057;color:#000;font-weight:bold;'
        f'border-right:1px solid #adb5bd;width:{round(100/len(model_topk), 2)}%;">'
        f'{esc(cfg["display_name"])}</th>'
        for cfg in (results[k]['cfg'] for k in model_topk.keys())
    )

    body = ''
    for rank in range(1, top_k + 1):
        cells = ''
        for key, rows in model_topk.items():
            if rank > len(rows):
                cells += '<td style="padding:10px;color:#000;border-right:1px solid #e9ecef;border-bottom:1px solid #e9ecef;vertical-align:top;">—</td>'
                continue
            r = rows[rank - 1]
            is_target = (r['text'].strip() == target_norm)
            bg = '#d4edda' if is_target else '#ffffff'
            border = '4px solid #28a745' if is_target else 'none'
            star = ' ★' if is_target else ''
            text_full = r['text'].replace('\n', ' ')
            cells += (
                f'<td style="padding:10px;color:#000;background:{bg};'
                f'border-left:{border};border-right:1px solid #e9ecef;'
                f'border-bottom:1px solid #e9ecef;vertical-align:top;font-size:12px;">'
                f'<div style="font-weight:bold;font-size:11px;margin-bottom:4px;color:#495057;">'
                f'#{rank}{star} · @{esc(r["channel"])} · {esc(r.get("category",""))}</div>'
                f'<div style="line-height:1.4;color:#000;">{esc(text_full)}</div>'
                f'</td>'
            )
        body += f'<tr>{cells}</tr>'

    return HTML(f'''
    <div style="border:2px solid #495057;border-radius:8px;margin:32px 0;
                background:#ffffff;font-family:system-ui,sans-serif;overflow:hidden;color:#000;">
        <div style="background:#e9ecef;padding:14px 20px;border-bottom:1px solid #ced4da;">
            <div style="font-size:17px;font-weight:bold;color:#000;">
                Пара {idx}/{total} · GT post #{pair["post_num"]}
            </div>
            <div style="font-size:13px;color:#495057;margin-top:4px;">
                {esc(pair.get("imt_name",""))} · {esc(pair.get("subj_name",""))}
            </div>
        </div>
        <div style="padding:14px 20px;background:#e7f3ff;border-bottom:1px solid #cfe2ff;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Запрос (описание товара)</div>
            <div style="color:#000;font-size:13px;line-height:1.5;">{esc(pair["description"])}</div>
        </div>
        <div style="padding:14px 20px;background:#d4edda;border-bottom:1px solid #c3e6cb;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Целевой пост (должен попасть в top-{top_k})</div>
            <div style="color:#000;font-size:13px;line-height:1.5;white-space:pre-wrap;">{esc(target_text)}</div>
        </div>
        <table style="width:100%;border-collapse:collapse;background:#ffffff;table-layout:fixed;">
            <thead><tr>{col_headers}</tr></thead>
            <tbody>{body}</tbody>
        </table>
    </div>
    ''')


In [11]:
# ============================================================
# ВИЗУАЛ: для каждой из GT пар — топ-5 по каждой модели
# Это качественный контроль, НЕ метрика. Нужен чтобы глазами увидеть,
# что модели вытаскивают осмысленные посты (даже если помеченный как
# 'правильный' пост не всегда оказывается в топ-5).
# ============================================================
for idx, pair in enumerate(gt_pairs, 1):
    target_text = post_text_by_num[pair['post_num']]
    model_topk = OrderedDict()
    for cfg in MODELS:
        model = loaded_models[cfg['key']]
        table = db.open_table(cfg['table_name'])
        q_in = (cfg['query_prefix'] + pair['description']) if cfg['query_prefix'] else pair['description']
        qvec = model.encode([q_in], normalize_embeddings=True)[0].tolist()
        rows = (table.search(qvec, query_type='vector')
                     .limit(TOP_K_VISUAL)
                     .select(['text', 'channel', 'category'])
                     .to_list())
        model_topk[cfg['key']] = rows
    display(render_pair_topk(idx, len(gt_pairs), pair, target_text, model_topk, top_k=TOP_K_VISUAL))


USER-bge-m3 base
"#1 · @baranka0707 · Семья и детиМоя находка на ВБ Пластины для стирки WiseНаконец-то больше не нужно носить тяжелые пакеты с порошком и кондиционером из магазина Никогда бы не подумала, что такие пластины будут справляться с загрязнениями не хуже порошка А самое главное, очень приятная цена))В упаковке 20 пластин - это до 40 стирок Какие плюсы отметила для себя: удобны в применении - не надо: лить, сыпать, отмерять не нужно носить тяжелые мешки с магазина, которые занимают много места дома после стирки вещи мягкие и приятные к телу имеет приятный свежий аромат, при этом не содержит интенсивных отдушек, фосфатов и парабенов, а значит не вызывает аллергических реакций и не загрязняет окружающую среду ребенок не разольет/не просыпетПереходите по ссылке и заказывайте, пока они есть в наличии Очень крутые 150669762"
"#2 · @elenazimenkova · Семья и детиFaberlic Home Как же я ее ждала! В Faberlic появилась новая серия посуды из нержавеющей стали Давно хотела глубокую сковородку (артикул 910440), ведь в ней можно готовить без воды и без масла, используя собственную влагу продуктов. Влага свободно циркулирует внутри емкости, пар почти не выходит, а продукты тушатся в собственном соку Тройное термоаккумулирующее дно Подходит для всех видов плит, включая индукционные (у меня как раз индукция) Размер: диаметр – 24 см, высота – 6 см. Объем: 2,7 л.Она великолепна!Крышка стеклянная с отверстием для выхода пара, корпус с делениями для определения объема пищи, пластиковые ручки. Экологически чистая, сохраняющая витамины, экономящая газ/электричество, масло, безвредная для здоровья посуда Ну и конечно, хочется отметить стильный лаконичный внешний вид. В ней готовить одно удовольствие"
"#3 · @skidky7 · ПродажиСохрани на WB # товары для дома Цена : 668 рублей — это 100 стирок Уже можно не переплачивать за Tide и Ariel 4.7 Только честные отзывы Отстирает на ура, жирные пятна, выполаскивание в холодной воде, парфюмированная добавка,которая заменит кондиционер, оставляя слегка уловимый аромат, без химозного запаха, подходит для детей Еще, этот порошок используют в сети отелей, но под другим брендом Смотреть тут"
"#4 · @posudakukmara · Еда и кулинарияКрасота, стиль, долговечность – эти эпитеты идеально подходят для описания нашей новой коллекции текстиля ""Правила кухни"". Комплект выполнен из плотного материала - рогожки, стойкого к истиранию. Он не абсорбирует запахи, мгновенно поглощает жидкости и быстро сохнет, не линяет после стирки и легко гладится.Казалось бы, этот комплект уже идеален, да? А если мы скажем, что с ним прекрасно сочетается набор кухонных принадлежностей из силикона, Ваша любовь к готовке станет ещё сильнее.Вот, теперь Вы непременно их хотите!? В коллекциях 3 цвета - подбирайте самый подходящий вариант для себя или в подарок на сайте"
"#5 · @zagorod_live · Интерьер и строительствоСУПЕР СРЕДСТВО УДАЛЯЕТ ЧЕРНУЮ ПЛЕСЕНЬ и ГРИБОК за 30 минут Фунгицид в составе обладает дезинфицирующим и отбеливающим эффектом для борьбы с плесенью, грибком и бактериями на различных поверхностях: керамических, пластиковых, бетонных, деревянных и окрашенных. Отличное средство для чистки грибка в стиральной машине. Средство для удаления от плесени и грибка идеально для применения в местах повышенной влажности: от плесени в ванной и на кухне. Цена: 549₽Артикул: 150118120"


USER-bge-m3 base
#1 · @dezigas · Интерьер и строительствоКомпактная настенная сушилка для белья Она предельно удобна для размещения в ванной или на балконе. Закрепив сушилку на стене один раз Вы больше не будете испытывать вопросов куда убрать сушилку - она всегда на своём местеДизайн интерьера
#2 · @dezigas · Интерьер и строительствоКомпактная настенная сушилка для белья Она предельно удобна для размещения в ванной или на балконе. Закрепив сушилку на стене один раз Вы больше не будете испытывать вопросов куда убрать сушилку - она всегда на своём местеДизайн интерьера
"#3 · @znaet_petrovich · Интерьер и строительствоНекоторые производители выпускают коллекции фоновых и акцентных обоев, которые сочетаются между собой. Если же готовые варианты вас не устраивают — есть несколько правил, которые помогут выбрать обои-компаньоны и выделить стены разными фонами и рисунками.Собрали полезные дизайнерские приемы в статье: znaet.petrovich.ru/oboi-kompanony-kak-ih-pravilno-sochetat"
"#4 · @znaet_petrovich · Интерьер и строительствоНекоторые производители выпускают коллекции фоновых и акцентных обоев, которые сочетаются между собой. Если же готовые варианты вас не устраивают — есть несколько правил, которые помогут выбрать обои-компаньоны и выделить стены разными фонами и рисунками.Собрали полезные дизайнерские приемы в статье: znaet.petrovich.ru/oboi-kompanony-kak-ih-pravilno-sochetat"
"#5 · @remont_na5 · Интерьер и строительствоЩелевой напольный плинтус - это маленький, едва заметный, напольный профиль со специальными креплениями, которые работают по принципу пружин и вставляются в напольное покрытие. Он устанавливается горизонтально, в одной плоскости с полом, основная функция - скрыть зазор между напольным покрытием и стеной.Ремонт на 5"


USER-bge-m3 base
"#1 · @MARKETSALEEE · Мода и красотаРучки пиши стирай#КанцелярияЦена: 339₽ 540₽ (Скидка -37%)Гелевые ручки пиши стирай одни из уникальных аксессуаров в письменной канцелярии. Стираемые ручки относятся к типу гелевых ручек со схожим наполнителем. Ручка пишет мягко, не царапает, чернила высыхают за 1 - 3 секунды, что гарантирует чистое письмо не оставляя разводов на бумаге. В комплекте с ручкой идут 50 запасных стержней. Это находка для школьников и студентов. Ссылка на товар"
"#2 · @mom_lena_channel · Семья и детиНадюша у нас очень любит читать книги: и себе, и нашим младшим)) А сейчас вышли такие новинки, мимо которых мы просто не могли пройти И уже в восторге) Серия книг легендарного издательства «Малыш» в сотрудничестве с киностудией «Союзмультфильм».- Книги по самым популярным мультикам «Союзмультфильма» с очень интересными текстами и рисунками из мультфильмов.- крупный шрифт, что очень важно для хорошего зрения.- диалоги в виде забавных комиксов и цветные иллюстрации, что так любят дети! Книги отлично подойдут для первого чтения. Они заинтересуют малышей, которые только начинают или не любят читать, станут отличной альтернативой гаджетам и мультфильмам, особенно летом! А ещё в этом году в прокат вышли полнометражные современные экранизации по мотивам Союзмультфильма. Отличный повод пересмотреть любимые мультфильмы и прочитать вместе с ребёнком эти яркие книги! Очень рекомендую!Вашим детям точно понравится)) Заказать можно тут!Реклама. ООО ""Издательство АСТ"" ИНН 7710899593 erid: 2VSb5xhPrHn"
"#3 · @podelki_deti · Семья и детиПроявляющиеся рисунки Игра с такиии рисунками несомненно увлечёт детей, и займёт их на долгие время, дав маме шанс спокойно попить чай Детские поделки | #бумага"
"#4 · @podelki_deti · Семья и детиПроявляющиеся рисунки Игра с такиии рисунками несомненно увлечёт детей, и займёт их на долгие время, дав маме шанс спокойно попить чай Детские поделки | #бумага"
"#5 · @mummyshcool · Семья и детиЗанятия с лепкой На сегодняшний день есть много различных масс для лепки: пластилин, пластичные массы, легкий или воздушный пластилин, разноцветное тесто. Все это можно купить или сделать самим в домашних условиях (рецепты приготовления под постом, сохраняйте).Из пластилина можно не только лепить фигуры, но и рисовать им, мять, размазывать. Подходит даже для самых маленьких творцов. Например, для деток в 1,5 года можно нарисовать простые и понятные шаблоны: солнышко , деревце , ягодки …С ПОМОЩЬЮ МАСС ДЛЯ ЛЕПКИ: Разнообразим досуг ребенка Подготовим ручки малыша к освоению навыков рисования, а в дальнейшем письма Совершенствуем моторику Учим или закрепляем цвета Развиваем творчество и воображениеВАЖНО хвалим ребенка независимо от результата лепки пластилин и тесто для лепки нужно разогреть перед использованием (слишком твердая масса плохо поддается еще слабым ручкам) для начала лучше применять варианты воздушного пластилина, а по мере освоения предлагать более плотные массы (так не пропадет интерес к занятию) лепить совместно с родителями в целях безопасности"


USER-bge-m3 base
"#1 · @katyabux · БлогиНе могу пройти мимо всяких интересных штучек для детей. Смотрите , что нашла в Фаберлик- детская мыльная краска для купания. Взяла все цвета, что были в наличии, это зеленая краска, розовая и желтая) Еще и с сочными фруктовыми ароматами - клубника, ананас и банан. Такая пенка позволяет ребенку рисовать и проявлять фантазию прямо в ванной, и создает легкую пенку)"
"#2 · @katyabux · БлогиНе могу пройти мимо всяких интересных штучек для детей. Смотрите , что нашла в Фаберлик- детская мыльная краска для купания. Взяла все цвета, что были в наличии, это зеленая краска, розовая и желтая) Еще и с сочными фруктовыми ароматами - клубника, ананас и банан. Такая пенка позволяет ребенку рисовать и проявлять фантазию прямо в ванной, и создает легкую пенку)"
"#3 · @facktoriym · ПозновательноеЯркий и необычный подарок, который понравится абсолютно всем Развивает воображение, мелкую моторику и усидчивость! Увлекает надолго как взрослого, так и ребенка. Готовая работа классно впишется в любой интерьер ЦЕНА: 815 руб."
"#4 · @facktoriym · ПозновательноеЯркий и необычный подарок, который понравится абсолютно всем Развивает воображение, мелкую моторику и усидчивость! Увлекает надолго как взрослого, так и ребенка. Готовая работа классно впишется в любой интерьер ЦЕНА: 815 руб."
"#5 · @towary_wildberriess · Видео и фильмыКонструктор интерьерный3D пазлы - это удивительный набор для творческого развития детей с 7 лет и взрослых. Мозайка включает большое количество пластмассовых деталей разных цветов, что позволяет создавать фигуру. Он идеально подходит как для девочек, так и для мальчиков.Перейти к каналу"


USER-bge-m3 base
"#1 · @kitaiznae · ПродажиХочешь убрать лишний вес?Забудь о таблетках и жёстких диетах!!! Проходи тест Получай готовый рацион питания Кушай вкусноИндивидуальные планы питания, чтобы похудеть быстро.Ешьте хорошо, выглядите потрясающе!Тест - для всех!!!"
"#2 · @kitaiznae · ПродажиХочешь убрать лишний вес?Забудь о таблетках и жёстких диетах!!! Проходи тест Получай готовый рацион питания Кушай вкусноИндивидуальные планы питания, чтобы похудеть быстро.Ешьте хорошо, выглядите потрясающе!Тест - для всех!!!"
"#3 · @kitaiznae · ПродажиХочешь убрать лишний вес?Забудь о таблетках и жёстких диетах!!! Проходи тест Получай готовый рацион питания Кушай вкусноИндивидуальные планы питания, чтобы похудеть быстро.Ешьте хорошо, выглядите потрясающе!Тест - для всех!!!"
"#4 · @TrenerBrin · Здоровье и фитнесНовая жиросжигающая программа, под кодовым названием «Сила-Творожка». Гарантированный результат -12кг жира! Бесплатно скачать можно вот тут: связи по ту сторону. Брин."
"#5 · @fitnes_yoga_doma · Здоровье и фитнесЖиросжигающий коктейль с имбирёмСостав:— стакан кефира (жирность 0,1-2,5%)— половина столовой ложки корицы,— половина столовой ложки имбиря— щепотка красного перцаПриготовление:Всё перемешивается.Рекомендуется выпивать с утра и перед сном.Регулярное употребление в течение месяца поможет избавиться от 3-4 кг веса."


...

USER-bge-m3 base
"#1 · @favouriteWB24 · Мода и красотаС такой подушкой каждое утро будет добрым!Подушка с эффектом памяти Поможет расслабить мышцы шеи и спины и занять правильное положение во время сна. Чехол снимается, а два валика помогут подобрать комфортную высоту подушки под себя. Цена сейчас: 1869₽ Обычная цена: 7331₽ Успей заказать"
"#2 · @deep_fake_bot_news · ТехнологииЯпонцы создали подушку для комфортного сна в офисеУстройство помогает выспаться прямо за столом без напряжения в шее и онемения рук. Регулируется угол наклона, а встроенный таймер разбудит приятной вибрацией (максимальный срок сна — 60 минут).Продажи стартуют 15 октября, на предзаказ уже выстроилась очередь. Стоимость сонного гаджета — $60 Дипфейкер Наш бот для дипфейков"
"#3 · @deep_fake_bot_news · ТехнологииЯпонцы создали подушку для комфортного сна в офисеУстройство помогает выспаться прямо за столом без напряжения в шее и онемения рук. Регулируется угол наклона, а встроенный таймер разбудит приятной вибрацией (максимальный срок сна — 60 минут).Продажи стартуют 15 октября, на предзаказ уже выстроилась очередь. Стоимость сонного гаджета — $60 Дипфейкер Наш бот для дипфейков"
#4 · @wb_ozon_sale_skidki · Мода и красотаПодушкаЦена: 656₽ (вместо 4 005₽)#товарыдо1000 Анатомическая подушка 50х70 сочетает в себе функциональность и комфорт. Она обладает антистрессовым эффектом и помогает справиться с храпом благодаря специальной технологии антихрапа. Создана из натуральных материалов: наполнитель - микроволокно и чехол из микрофибры. Изделие является гипоаллергенным и подходит даже для самой чувствительной кожи.Ссылка:
"#5 · @htech_plus · ТехнологииПовязка Elemind быстро погружает в сон без лекарствУченые из Массачусетского технологического института представили новое устройство в виде повязки для головы, которое активирует в мозге режим «шумоподавления» и помогает заснуть быстро и без лекарств. Elemind появится в продаже в США уже этим летом."


USER-bge-m3 base
"#1 · @ostrovok_travel · ПутешествияЗимние путешествия порой наталкивают на бесконечную череду мыслей «слишком холодно», «жарко», «надо было всё-таки взять джемпер» и «зачем я надел тёплый пуховик»... Вместе с нашими друзьями из TJ Collection делимся советами, что положить в чемодан, чтобы и ощущать себя комфортно, и при этом не отстать от моды!Реклама. ООО «КЛ ГРУПП». Erid: 2VtzqvbM2cj"
"#2 · @wildberries_do_100 · ПродажиСтильная и практичная, идеально подойдет для ноутбука и документов на работу или учебу! Обеспечит надежную защиту Вашего устройства от трещин и царапин благодаря внутреннему отделению с мягкой подкладкой. Наружная часть выполнена из водоотталкивающего материала. Имеется множество дополнительных кармашков для всех нужд. Рекомендуем!Cсылка"
"#3 · @nakhodki_Wildberries0 · ПродажиСтильная и практичная, идеально подойдет для ноутбука и документов на работу или учебу! Обеспечит надежную защиту Вашего устройства от трещин и царапин благодаря внутреннему отделению с мягкой подкладкой. Наружная часть выполнена из водоотталкивающего материала. Имеется множество дополнительных кармашков для всех нужд. Рекомендуем!Cсылка"
"#4 · @wildberries_do_100 · ПродажиСтильная и практичная, идеально подойдет для ноутбука и документов на работу или учебу! Обеспечит надежную защиту Вашего устройства от трещин и царапин благодаря внутреннему отделению с мягкой подкладкой. Наружная часть выполнена из водоотталкивающего материала. Имеется множество дополнительных кармашков для всех нужд. Рекомендуем!Cсылка"
"#5 · @SumkiBijuteryotRuslana · Мода и красотаДорожная сумка-это идеальный выбор не только для любителей спорта.Её можно взять с собой не только на тренировку в зал или бассейн,но и в путешествие. Она идеальна для похода в спортзал или для поездки, в качестве ручной клади в самолёт.Непромокаемая сумка изготовлена из качественной водонепроницаемой ткани, что обеспечивает долговечность и защиту ваших вещей.У неё вместительное внутреннее отделение для хранения спортивной формы и сменной одежды, отдельный боковой карман для обуви, а также наружные и внутренние карманы для размещения аксессуаров . Большая сумка также оснащена удобными ручками и регулируемыми плечевым ремнём для комфортного ношения через плечо в дороге.Опт 500₽ роз 600₽Размер 45#27#20"


USER-bge-m3 base
"#1 · @ribalka_prostaya · ПриродаПодписчик из нашего чата делится своим уловом Саратовская область, Петровский район.Первая на 670грВторая на 4 кг.Всем НХНЧ Простая Рыбалка l Наш чат"
"#2 · @AndreyPitertsov · ПриродаПрошло уже два полноценных сезона, как я активно гоняю крупные приманки спиннинговым комплектом. Палочки в моих руках вы могли видеть разные: Хайрон до 140 или 200 г. Хелл Хаунд до 160... А вот мясорубка всегда одна - Твин 20 года 4000pg.Знаю, что многим интересно как она? Что с ней случилось после достаточно продолжительных нагрузок, не развалилась ли?К удивлению для многих скажу, что с ней все хорошо. Ход стал немного более грубым - это единственное что произошло, и это нормально. Больше и добавить нечего. Громыхать, урчать, скрипеть не начала.Так что практикой удалось подтвердить то, что ""мясорубочный"" комплект отлично подходит для больших приманок. Далее каждый сам решает на что ему ловить приятнее и комфортнее (мульт или мясорубка)."
"#3 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
"#4 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
#5 · @Luxbags116 · ПродажиСНОВА В НАЛИЧИИ МУЖСКИЕ ПОРТМАНЕ В идеальном качестве Канва натуральная На 2 молния В комплекте коробка и пыльник Размер: 21/11 см Цена : 2800₽


USER-bge-m3 base
"#1 · @ribalka_sovet · ПозновательноеТЕХАССКАЯ ОСНАСТКА — ИЗГОТОВЛЕНИЕ И ТЕХНИКА ЛОВЛИ Техасская оснастка относится к разряду абсолютных «незацепляек» и своим происхождением обязана одному из озер в штате Техас. Изначально оснастка применялась для ловли американского басса, но в силу своей универсальности и уловистости она завоевала популярность и у наших спиннингистов. В условиях наших водоемов Техасская оснастка с успехом применяется для ловли щуки, окуня, судака и даже язя. Что представляет собой Техасская оснастка и где применяется? Техасская оснастка устроена достаточно просто — офсетный крючок с насаженным на него силиконовым червем привязан к основной леске, на которой скользит грузило в виде пули. Между крючком и пулей устанавливается бусинка (бисер). Хотя ее наличие не всегда обязательно, но об этом позже."
"#2 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
"#3 · @AndreyPitertsov · ПриродаВ прошлом посте обсуждали оптимальное оснащение отводного поводка...Сейчас расскажу про две основные схемы, которые я использую.1. Она на втором фото. Люблю этот вариант, когда часто меняю джиг и отводной. Вертлюг с поводком там просто вставляю в верхнее ухо стального поводка. Чтобы опять вернуться к джигу, просто раскручиваю скрутку, снимаю поводок, а вместо груза вешаю джиг. Очень быстро и удобно. Один минус - путается немного чаще, чем второй вариант.2. Смотрим на рисунок. Я конечно рукожоп в этом плане) ну тут главное смысл, а не красота. Лесочный поводок с грузом на вертлюге просто скользит по основной леске и упирается во второй вертлюг. Если долго ловлю именно отводным и менять не планирую, тогда это лучшая оснастка. Практически нереально запутать.Оптимальная длина поводка с грузиком 20-40 см, поводка с приманкой 100-120 см."
"#4 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum."
"#5 · @rezatribu · ПозновательноеFeeder Gum Это отрезок резины, который создает амортизацию и гасит резкие рывки рыбы.Вяжется он между фидерной оснасткой и очень тонким флюрокарбоновым поводком (например, 0.1 мм).Используется длина 10-12 см и толщина 0.6-0.8 мм.Если требуется деликатное вываживание при очень тонкой леске (0.1 мм), то применяют именно Feeder Gum."


USER-bge-m3 base
"#1 · @valberis_odezhda1 · Мода и красотаЗабудьте о холодной и неприятной зиме с этой прекрасной шапкой. Неважно, идете ли вы на прогулку по зимнему парку или отправляетесь на работу, шапка обеспечит вам комфорт и тепло на протяжении всего дня. Вы можете быть уверены, что ваша голова будет защищена от суровых погодных условий. Утеплиться"
#2 · @FakFashion · Мода и красотаВыходим на облегченные весенние лучки. Собрала вариант повседневного образа на большинство жизненных обстоятельств. Пиджак Футболка Джинсы Солнцезащитные очки Резинки для волосСумка Кроссовки
"#3 · @odevaysakrasiva · Мода и красотаВпереди еще одни майские праздники, а это значит, что многие из вас отправятся на пикники с семьей или близкими друзьями. Чтобы выглядеть на природе стильно, но при этом уместно, в первую очередь важно подобрать правильную обувь. Например, резиновые сапоги, которым не страшны грязь и весенние дожди. Что приятно, носить их можно не только за городом, но и в мегаполисе — доказано лучшими стилистами, которые сочетают сапоги как со строгими пиджаками и женственными платьями, так и со свитерами, джинсами и дождевиками."
"#4 · @frontroww · Мода и красотаКак-то совсем не уделили внимание еще одному выигрышному тренду этой осени. Юбки-шотланки уже тепло пригрелись в гардеробах фешн-инфлюенсеров, да и нам такие образы оказались очень даже близки.Кажется, такая юбка смотрится идеально буквально с любым элементом: начиная от облегающего лонглслива, заканчивая грубым пальто или дутой курткой. Ну и по традиции, делимся вариантами стилизации!"
"#5 · @Ozon_sale_promocode · ПродажиУютные аксессуары: что надеть, чтобы не замёрзнуть.Поиск подходящего фасона шапки — это миссия, по сложности сопоставимая с поисками идеального оттенка красной помады. Так, может, и не стоит тратить время? Ведь для того, чтобы согреться, есть и другие аксессуары: снуды, капюшоны или капоры, которые в этом сезоне на пике моды.Сделали подборку аксессуаров для тех, кто не хочет мёрзнуть этой осенью. Вязаная манишка Жилет утеплённый Косынка Снуд Капор утеплённый Шерстяной капор"


In [12]:
# Освобождаем VRAM
for k in list(loaded_models.keys()):
    del loaded_models[k]
loaded_models.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Память освобождена.')


Память освобождена.
